# Streaming Sub-Agent Updates

Demonstrates how to stream tool calls and responses from a **SubAgent** hierarchy.

**Features covered:**
- Defining a `SubAgent` with custom tools
- Streaming with `stream_mode=["messages", "updates"]` and `subgraphs=True`
- Tracking which agent is currently responding via `lc_agent_name` metadata
- Setting the `name` parameter for better trace identification

**Prerequisites:**
- `deepagents`, `langgraph`, `langchain-openai`, `python-dotenv` packages
- `OPENAI_API_KEY` environment variable (loaded from `~/.env/orchestra/.env.backend`)

In [ ]:
!uv pip install -q langgraph langchain-openai deepagents python-dotenv

## Environment Setup

Load API keys from the standard Orchestra env file.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from standard location
# ~/.env/orchestra/.env.backend
env_path = Path.home() / ".env" / "orchestra" / ".env.backend"
load_dotenv(env_path)

## Define Sub-Agent and Stream Responses

Create a weather sub-agent, wire it into the main agent, and stream the full interaction including sub-agent tool calls.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessageChunk, AIMessage, ToolMessage, AnyMessage
from deepagents import create_deep_agent, SubAgent

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")

def get_weather(city: str) -> str:
    """Get the weather in a given city."""
    return f"The weather in {city} is sunny."

weather_agent = SubAgent(
    name="weather_agent",
    description="Get the weather in a given city.",
    model="openai:gpt-4.1-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

agent = create_deep_agent(
    name="subagent-streaming-demo",
    model="openai:gpt-4.1-mini",
    subagents=[weather_agent],
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver()
)

input_message = HumanMessage(content="Weather in Dallas?")

current_agent = None
for _, stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
    config={"configurable": {"thread_id": "subagent-tool-calls"}},
    subgraphs=True,
):
    if stream_mode == "messages":
        token, metadata = data
        if agent_name := metadata.get("lc_agent_name"):
            if agent_name != current_agent:
                print(f"🤖 {agent_name}: ")
                current_agent = agent_name
        if isinstance(token, AIMessage):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])